In [ ]:
import datetime
import os
from src.ingestion.openweather import fetch_weather_history
from src.ingestion.loaders import load_json_to_spark_df
from src.utils.config import *
from src.utils.spark import get_spark

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

api_key = "YOUR_OPENWEATHER_KEY" # replace later with os.getenv('WEATHER_API_KEY')

column_names = ['weather_time', 'main', 'wind', 'clouds', 'rain', 'snow', 'weather']
defaults = {
  'weather_time': 0,
  'main': {},
  'wind': {},
  'clouds': {},
  'rain': {},
  'snow': {},
  'weather': []
}

records = fetch_weather_history(
    api_key=api_key,
    lat=1.290270,
    lon=103.851959,
    start_date=datetime.datetime(2026, 2, 1),
    end_date=datetime.datetime(2026, 3, 1),
)

In [ ]:
weather_df = load_json_to_spark_df(spark, records)
weather_df.show(5)

In [ ]:
output_path = "/content/drive/MyDrive/aviation-delay/data/raw/weather.csv"

weather_df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")

In [ ]:
# kafka loading from csv
BOOTSTRAP_SERVERS = ['localhost:9092']
hist_we_prod = create_kafka_producer('historical_weather_data', BOOTSTRAP_SERVERS)
hist_we_cons = create_kafka_consumer('consume_weather_data', BOOTSTRAP_SERVERS, 'raw_weather_data')

# give the data to kafka topic to process
stream_to_kafka(output_path, hist_we_prod, 'raw_weather_data')
# insert data to clickhouse for storage
insert_to_clickhouse(hist_we_cons, 'historical_weather_data', client, column_names, defaults)